# APM_50441_EP - Machine Learning 2 - Lab 6: Reinforcement learning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APM-50441-EP/APM_50441_EP-2026-2027/blob/main/PC6/APM_50441_EP_Lab_6.ipynb) or run it on your own computer (see *Working environment* below).

This lab illustrates the sixth lecture (*Reinforcement Learning*) on the Cliff Walking problem of the lecture. Its goals are:
- to describe a Markov decision process and its model, and to solve it by dynamic programming (policy evaluation, value iteration, policy iteration),
- to compare Monte Carlo and temporal difference estimation of a value function,
- to implement SARSA and Q-learning, and to understand why they learn different paths,
- to implement the REINFORCE policy gradient in PyTorch, and to see the effect of the reward-to-go on its variance.

The optional *To go further* part, after the core, can be skipped at first: it covers bandits, more temporal difference methods, the deadly triad, deep RL (DQN, actor-critic, PPO) and the RL of language models (GRPO, DPO, agents).

We use `Python` with `numpy`, `gymnasium` and `PyTorch`. The environment, its tabular model and drawing helpers are in the small module `RL.py` (no algorithm in it: you write them).

**Working environment.** Colab is the simplest option: open the notebook of this lab from the course repository (*Open in Colab* link) and run it. To work on your own computer, download the lab folder (the notebook and its files) or clone the course repository, and open the notebook in an IDE that runs Jupyter notebooks (VS Code with the Python and Jupyter extensions, JupyterLab, Spyder, PyCharm). Create a Python environment once with [`uv`](https://docs.astral.sh/uv/): `uv venv`, activate it (`source .venv/bin/activate`, or `.venv\Scripts\activate` on Windows) and `uv pip install jupyter`, or build the whole course environment with `uv sync` in `PC/env`; then select this environment as the kernel of the notebook. The installation cell below installs the packages of the lab into the Python running the notebook (Colab or your environment), and the data are read from the local files when they are present (otherwise they are downloaded).

**Use of AI.** AI assistants are allowed, but you must be able to explain every line you use, and you should answer the *predict before you run* questions before running the code: the suggested code may not be correct!

In [ ]:
# Installs into the Python running this notebook: Colab, or locally an activated virtual environment
import sys
!uv pip install --python "{sys.executable}" "gymnasium[toy-text]==1.2.*" "torch" "polars==1.44.2" "seaborn==0.13.2"

In [ ]:
import os
import urllib.request
if not os.path.exists("RL.py"):
  urllib.request.urlretrieve("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/PC6/RL.py", "RL.py")
import time
import numpy as np
import polars as pl
import seaborn.objects as so
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
torch.set_num_threads(2)
import RL
from RL import NS, NA, START, GOAL, VALID, UP, RIGHT, DOWN, LEFT

## 1. The environment

Cliff Walking (Sutton and Barto, Example 6.6, and the lecture) is a $4 \times 12$ grid: the agent starts in S (bottom left), must reach G (bottom right), and pays $-1$ per step; stepping into the cliff costs $-100$ and sends it back to S. The states are numbered $s = 12 \times \text{row} + \text{column}$ and the actions are $0$ = up, $1$ = right, $2$ = down, $3$ = left. A `gymnasium` environment is used through `reset()` and `step(action)`, which returns the next state, the reward, and whether the episode is *terminated* (goal reached) or *truncated* (time limit).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
RL.draw_grid(ax, paths=[(RL.OPTIMAL_PATH, 'tab:orange', 'optimal path (13 steps)'), (RL.SAFE_PATH, 'tab:blue', 'safe path (17 steps)')])

**Question** *Predict before you run:* what is the average return of the uniformly random policy, with episodes cut after $500$ steps? Run $200$ such episodes in `RL.make_env(max_steps=500)` and report the mean return, the mean length and the proportion of episodes reaching the goal.

**Answer**

---

The lecture also uses a *slippery* version: with probability $10\%$, the chosen action is replaced by a uniformly random one. `RL.tabular_model(slip)` returns its model: the transition probabilities `P[s, a, s']` and the expected rewards `R[s, a]` (the goal is absorbing, with reward $0$).

**Question** Build the model with `slip=0.1`. Check that each `P[s, a]` is a probability distribution. Where can the agent go from S when it chooses *right*, with which probabilities? Compute `R[START, RIGHT]` by hand and compare.

**Answer**

---

## 2. Dynamic programming

With a known model and a discount $\gamma = 0.9$, the value of a stationary policy $\pi$ solves the Bellman equation $v_\pi = r_\pi + \gamma P_\pi v_\pi$, with $P_\pi(s, s') = \sum_a \pi(a|s)P(s, a, s')$ and $r_\pi(s) = \sum_a\pi(a|s)R(s, a)$. It can be computed exactly by solving this linear system, or by iterating the Bellman operator $v \mapsto r_\pi + \gamma P_\pi v$, a $\gamma$-contraction for the sup norm (PC6). The optimal value $v_\star$ is the fixed point of the *optimal* Bellman operator $v \mapsto \max_a\big(R(\cdot, a) + \gamma P(\cdot, a)v\big)$.

**Question** For the uniform policy on the slippery grid ($\gamma = 0.9$), compute $v_\pi$ exactly with `np.linalg.solve`, then by $60$ iterations of the Bellman operator from $v = 0$. Plot the sup-norm error (over the valid states) along the iterations in log scale, with the bound $\gamma^k\|v_0 - v_\pi\|_\infty$. Display $v_\pi$ on the grid.

**Answer**

---

**Question** Implement value iteration (iterate the optimal Bellman operator from $v = 0$ until the sup-norm change is below $10^{-10}$). Display $v_\star$ with the greedy policy $\arg\max_a\big(R(s, a) + \gamma P(s, a)v_\star\big)$ and its path from S (without slips). Then solve the problem without slips (`slip=0`) and compare the paths.

**Answer**

---

**Question** Implement policy iteration: start from the policy "always right" (which walks into the cliff), evaluate it exactly, take the greedy policy, and repeat until the policy does not change. How many iterations are needed? Plot the sup-norm error $\|v_{\pi_k} - v_\star\|_\infty$ along the iterations together with the one of value iteration.

**Answer**

---

## 3. Monte Carlo and temporal difference estimation

Without a model, the value of a policy is estimated from episodes. *Monte Carlo* (MC) moves $V(s_t)$ towards the observed return $G_t$; *temporal difference* TD(0) moves it towards the *bootstrapped* target $r_{t+1} + \gamma V(s_{t+1})$, after each step. We compare them on the random walk of the lecture (Sutton and Barto, Example 6.2): $5$ states A to E, start in C, left or right with probability $1/2$, reward $1$ when leaving on the right; the true values are $1/6, \dots, 5/6$. `RL.random_walk_episode(rng)` returns the transitions $(s, r, s')$ of an episode ($s' = $ `None` at the end).

**Question** Implement every-visit MC and TD(0) with a constant step size $\alpha$ and the values initialized at $0.5$ (no discount). For $\alpha \in \{0.01, 0.02, 0.05, 0.1, 0.2, 0.3\}$, compute the RMS error over the $5$ states after each of $100$ episodes, averaged over $100$ runs. Plot the curves of the best $\alpha$ of each method, and the final error (mean over the last $10$ episodes) as a function of $\alpha$.

**Answer**

---

**Question** With a constant step size, the estimates keep fluctuating. Compare, over $1000$ episodes ($20$ runs), MC and TD(0) with $\alpha = 0.05$ and with the decreasing step size $\alpha_n = 1/n$ ($n$ = number of updates of the state). Which ones converge to the true values? (PC6, Exercise 2.)

**Answer**

---

## 4. SARSA and Q-learning

Without a model, control learns the action values $Q(s, a)$ from the transitions, while acting $\varepsilon$-greedily. After a transition $(s, a, r, s')$ and the choice of the next action $a'$:
- SARSA (on-policy) uses the target $r + \gamma Q(s', a')$: it learns the value of the $\varepsilon$-greedy policy it follows,
- Q-learning (off-policy) uses the target $r + \gamma\max_b Q(s', b)$: it learns the value of the greedy policy.

As in the lecture, we use $\varepsilon = 0.1$, $\alpha = 0.5$, no discount, and no slip.

**Question** *Predict before you run:* which method will collect more reward during learning? Implement both on `RL.make_env(max_steps=1000)` ($500$ episodes, ties broken at random, $Q$ initialized at $0$), average the reward per episode over $20$ runs, and plot it (smoothed over $10$ episodes). Draw the greedy paths of the learned $Q$ (averaged over the runs).

**Answer**

---

## 5. Policy gradient: REINFORCE in PyTorch

A policy gradient method optimizes a parametrized policy directly. With a tabular softmax policy $\pi_\theta(a|s) \propto e^{\theta_{s,a}}$, the REINFORCE estimator of $\nabla_\theta J(\theta)$ for an episode $(s_0, a_0, r_1, \dots)$ is $\sum_t G\,\nabla_\theta\log\pi_\theta(a_t|s_t)$, with either the total return $G = G_0$ or, using causality, the *reward-to-go* $G = G_t = \sum_{t' \geq t} r_{t'+1}$ (PC6). In PyTorch, it is the gradient of the loss $-\sum_t G\log\pi_\theta(a_t|s_t)$, computed by automatic differentiation.

To keep the episodes short, they start from a random valid state and are cut after $100$ steps (no discount, no slip). The quality of a policy is measured exactly with the model: `expected_return` computes the expected sum of rewards over $100$ steps from S (backward induction).

In [ ]:
P0, R0 = RL.tabular_model(slip=0.0)
STARTS = [s for s in range(NS) if VALID[s] and s != GOAL]
def expected_return(theta, horizon=100):
  pi = torch.softmax(theta, dim=1).detach().numpy()
  P_pi, r_pi = np.einsum('sa,sat->st', pi, P0), (pi * R0).sum(axis=1)
  v = np.zeros(NS)
  for _ in range(horizon):
    v = r_pi + P_pi @ v
  return v[START]

**Question** Write a function sampling an episode with the policy $\text{softmax}(\theta)$ (return the states, the actions and the rewards), and the REINFORCE loss with the full return or the reward-to-go. Train $\theta$ (initialized at $0$, plain SGD with step sizes $3\cdot10^{-4}$ for the full return and $10^{-3}$ for the reward-to-go) for $4000$ episodes, $3$ runs each, and plot `expected_return` every $100$ episodes. The optimum is $-13$.

**Answer**

---

**Question** At the initial policy ($\theta = 0$), sample $2000$ episodes and compute, for each of them, the coordinate $\partial/\partial\theta_{S,\text{up}}$ of the gradient of the REINFORCE objective (minus the gradient of the loss) with both estimators. Compare their means and standard deviations. Why is the reward-to-go legitimate, and why does it help?

**Answer**

---

## To go further

The following blocks are optional and independent: each of them only uses the objects defined in the core part. Their difficulty is marked by stars: ★ direct extension, ★★ new method, ★★★ longer or research-flavored.

| Block | Lecture | Stars |
|---|---|---|
| Multi-armed bandits | Bandits | ★★ |
| Monte Carlo control | Monte Carlo control | ★ |
| $n$-step TD | $n$-step methods and eligibility traces | ★★ |
| Maximization bias and double Q-learning | Q-learning, DQN overestimation | ★★ |
| Transfer to Taxi | Tabular methods | ★ |
| Baird's counterexample | The deadly triad | ★★ |
| Baseline and actor-critic | Policy gradient, actor-critic | ★★ |
| GRPO: group-relative advantages | LLMs: reasoning with verifiable rewards | ★★ |
| DQN on CartPole | Deep Q-networks | ★★ |
| PPO | Proximal policy optimization | ★★ |
| Preferences: reward model and DPO | RLHF and DPO | ★★ |
| A tiny agent with retrieval | The agent loop, RAG | ★★★ |

The packages used in these blocks (not needed for the core) are installed here:

In [ ]:
import sys
!uv pip install --python "{sys.executable}" "gymnasium[classic-control]==1.2.*" "stable-baselines3==2.9.0" "transformers"

### Multi-armed bandits (★★)

A bandit is an MDP with a single state: at each step, the agent pulls one of $K$ arms and receives a noisy reward. The *regret* $\sum_t(\mu_\star - \mu_{A_t})$ measures the cost of learning. $\varepsilon$-greedy explores at a constant rate (linear regret), UCB plays $\arg\max_a\hat\mu_a + \sqrt{2\log t/N_a}$ (optimism in the face of uncertainty) and Thompson sampling plays the best arm of a sample of the posterior.

**Question** On the $10$-armed Gaussian testbed (means $\mu_a \sim \mathcal{N}(0, 1)$, unit noise, horizon $2000$, $100$ runs), compare the cumulative regret of $\varepsilon$-greedy ($\varepsilon = 0.1$), UCB and Thompson sampling (prior $\mathcal{N}(0, 1)$ on each mean, known unit noise).

**Answer**

---

### Monte Carlo control (★)

Monte Carlo control alternates the evaluation of $Q$ by averaging the returns of complete episodes and an $\varepsilon$-greedy improvement. It needs no bootstrapping, but each update waits for the end of the episode.

**Question** Implement on-policy every-visit MC control on Cliff Walking ($\varepsilon = 0.1$, episodes from S cut after $200$ steps, $2000$ episodes), with $Q(s, a)$ updated towards the observed returns either by their average (step size $1/n$) or with a constant step size $\alpha = 0.1$. Which paths are learned? Compare with SARSA and Q-learning.

**Answer**

---

### $n$-step TD (★★)

$n$-step TD uses the target $r_{t+1} + \dots + r_{t+n} + V(s_{t+n})$: $n = 1$ is TD(0), $n = \infty$ is MC. Intermediate values often work best (Sutton and Barto, Figure 7.2).

**Question** On the random walk with $19$ states (`RL.random_walk_episode(rng, n=19)`, true values $(s+1)/20$), compute the RMS error over the first $10$ episodes (averaged over $100$ runs, values initialized at $0.5$) of $n$-step TD for $n \in \{1, 2, 4, 8, 16, 64\}$ as a function of $\alpha$.

**Answer**

---

### Maximization bias and double Q-learning (★★)

The target $\max_b Q(s', b)$ of Q-learning is a maximum of noisy estimates, hence biased upwards: $\mathbb{E}[\max_b \hat Q(b)] \geq \max_b\mathbb{E}[\hat Q(b)]$. Double Q-learning keeps two tables, selects the action with one and evaluates it with the other. In the small MDP of Sutton and Barto (Example 6.7), from A the action *right* ends the episode with reward $0$, while *left* leads to B, from which $10$ actions end the episode with a reward $\mathcal{N}(-0.1, 1)$: going left is a mistake.

**Question** Run Q-learning and double Q-learning ($\varepsilon = 0.1$, $\alpha = 0.1$, no discount, $300$ episodes, $1000$ runs) on this MDP, and plot the proportion of *left* actions in A along the episodes. Check first the bias numerically: the mean of the maximum of $10$ sample means of $5$ draws of $\mathcal{N}(-0.1, 1)$.

**Answer**

---

### Transfer to Taxi (★)

`Taxi-v3` is another tabular environment: $500$ states (taxi position, passenger location, destination) and $6$ actions (moves, pick up, drop off), reward $-1$ per step, $+20$ for a correct drop-off and $-10$ for an illegal pick-up or drop-off.

**Question** Adapt your Q-learning to `gym.make('Taxi-v3')` ($\varepsilon = 0.1$, $\alpha = 0.5$, $\gamma = 0.99$, $3000$ episodes), plot the reward per episode, and evaluate the greedy policy on $100$ new episodes.

**Answer**

---

### Baird's counterexample (★★)

The *deadly triad* (function approximation, bootstrapping and off-policy learning) can make TD diverge. Baird's counterexample (Sutton and Barto, Example 11.1): $7$ states, all rewards $0$ (so $v_\pi = 0$), $\gamma = 0.99$; the behavior policy goes to one of the $6$ upper states with probability $6/7$ and to state $7$ with probability $1/7$, the target policy always goes to state $7$ (importance ratio $7$ or $0$). The linear features are $v(i) = 2w_i + w_8$ for $i \leq 6$ and $v(7) = w_7 + 2w_8$: $w = 0$ is a perfect fit.

**Question** Run off-policy semi-gradient TD(0) ($\alpha = 0.01$, $w_0 = (1, 1, 1, 1, 1, 1, 10, 1)$, $1000$ steps) with these linear features and with tabular features (same initial values), and plot the largest $|v(s)|$ along the steps. Then run the linear version *on-policy* (transitions drawn from the target policy... and from the behavior policy, without importance ratio).

**Answer**

---

### Baseline and actor-critic (★★)

Subtracting a baseline $b(s_t)$ from the reward-to-go does not bias the policy gradient (PC6) and reduces its variance; with a learned value $V(s_t)$ as baseline, the advantage $G_t - V(s_t)$ measures whether the action was better than usual. An *actor-critic* replaces $G_t$ by the TD target $r_{t+1} + \gamma V(s_{t+1})$ and updates after each step.

**Question** With $\gamma = 0.99$, compare on Cliff Walking ($2000$ episodes, $3$ runs, same episodes as in the core part) the reward-to-go REINFORCE, REINFORCE with a tabular value baseline (step size $0.1$ for $V$), and the one-step actor-critic (critic step size $0.1$). Measure the policies with the exact discounted value from S.

**Answer**

---

### GRPO: group-relative advantages (★★)

To fine-tune language models with verifiable rewards, GRPO (DeepSeek, 2024) avoids learning a critic: it samples a *group* of $G$ answers to the same prompt and uses as advantage the reward normalized within the group, $A_i = (r_i - \bar r)/s_r$. RLOO uses $r_i - \frac{1}{G-1}\sum_{j\neq i}r_j$, an unbiased leave-one-out baseline. The same idea applies to our episodes: the "prompt" is the start state, the "answer" an episode.

**Question** Train the tabular softmax policy on Cliff Walking with groups of $G = 8$ episodes from the same random start state (each episode weighted by its group advantage, the same for all its steps), with the GRPO and the RLOO advantages, and with the raw return (the full-return REINFORCE of the core part), for $1000$ groups ($8000$ episodes, $2$ runs).

**Answer**

---

### DQN on CartPole (★★)

With a large or continuous state space, $Q$ is approximated by a neural network. DQN (Mnih et al., 2015) stabilizes the Q-learning of a network with an *experience replay* buffer (decorrelated mini-batches of past transitions) and a *target network* (a frozen copy, updated periodically, to compute the targets). `CartPole-v1`: keep a pole balanced on a cart (state of dimension $4$, $2$ actions, reward $+1$ per step, at most $500$ steps).

**Question** Implement DQN (network $4 \to 64 \to 64 \to 2$, Adam with learning rate $10^{-3}$, buffer of $10\,000$ transitions, mini-batches of $64$, $\gamma = 0.99$, $\varepsilon$ decreasing linearly from $1$ to $0.05$ over $10\,000$ steps, target network updated every $500$ steps) for $30\,000$ steps, and plot the length of the episodes.

**Answer**

---

### PPO (★★)

PPO (Schulman et al., 2017) reuses each batch of episodes for several gradient steps, with the *clipped* objective $\mathbb{E}\big[\min(\rho A, \text{clip}(\rho, 1 - \epsilon, 1 + \epsilon)A)\big]$, $\rho = \pi_\theta(a|s)/\pi_{\text{old}}(a|s)$: the policy has no incentive to move the probability ratio beyond $1 \pm \epsilon$. It is the algorithm of RLHF (Lecture 6), and a standard choice in control.

**Question** Plot the clipped objective as a function of $\rho$ for $A = 1$ and $A = -1$ ($\epsilon = 0.2$). Then train the PPO of `stable-baselines3` on `CartPole-v1` for $30\,000$ steps and evaluate it on $10$ episodes.

**Answer**

---

### Preferences: reward model and DPO (★★)

RLHF learns from comparisons: a human prefers answer $y_1$ to $y_2$ with probability $\sigma(r(y_1) - r(y_2))$ (Bradley–Terry). The classical pipeline fits a reward model on the preferences, then maximizes $\mathbb{E}_\pi[r] - \beta\,\text{KL}(\pi\|\pi_{\text{ref}})$, whose solution is $\pi_\star \propto \pi_{\text{ref}}e^{r/\beta}$ (PC6). DPO (Rafailov et al., 2023) plugs this solution into the Bradley–Terry likelihood and fits the policy *directly* on the preferences: $-\log\sigma\big(\beta\log\frac{\pi(y_w)}{\pi_{\text{ref}}(y_w)} - \beta\log\frac{\pi(y_l)}{\pi_{\text{ref}}(y_l)}\big)$.

**Question** Take a single prompt with $20$ possible answers, true rewards $r \sim \mathcal{N}(0, 1)$, a uniform reference policy and $\beta = 0.5$. Simulate $2000$ preferences between random pairs of answers. Fit (i) a reward model by logistic regression on the pairs, then the KL-regularized policy; (ii) the DPO policy by gradient descent on its loss (logits of a softmax policy). Compare both with $\pi_\star$ (total variation distance) and their expected true rewards.

**Answer**

---

### A tiny agent with retrieval (★★★)

An *agent* is a language model in a loop: at each step it can call a tool (a search engine, a calculator...), read the result, and decide what to do next, until it answers (lecture: *The Agent Loop*). *Retrieval-augmented generation* (RAG) puts the documents retrieved for the question in the prompt. We use a small open model (Qwen2.5-0.5B-Instruct, about $1$ GB, downloaded once; a GPU runtime on Colab is faster, and a laptop CPU takes a few minutes) with two tools: a retriever over a few notes on this lab (BM25-like scores) and a calculator. Small models are unreliable agents: the point is to see the loop, and its failures.

**Question** Run the agent loop below on the three questions. For each step, look at the model's output, the tool call and the observation. Then estimate the cost of an agent: if each step reads a context of $3000$ tokens and writes $200$ tokens, with prices of $1\$$ per million input tokens and $4\$$ per million output tokens, how much does a $10$-step task cost? If each step succeeds independently with probability $0.95$, what is the probability that the $10$ steps all succeed?

**Answer**

---

## About this lab

**Use of AI.** Artificial intelligence tools have been used to revise the content of this lab. The main contributor takes full responsibility for this content.

**License.** The texts of this lab are under the [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license (CC BY-SA 4.0), and its code under the [MIT](https://opensource.org/license/mit) license; the datasets keep their own licenses. Main contributor: E. Le Pennec. Contributors: S. Boucheron, A. Dieuleveut, A.K. Fermin, S. Gadat, S. Gaiffas, B. Goujaud, A. Guilloux, Ch. Keribin, Th. Klein, E. Matzner, J. Read, M. Sangnier, E. Scornet.